# Projeto: Coleta e Análise de Narrativas de Startups

## CÉLULA 0 — INSTRUÇÕES

Para que o código funcione corretamente, você deve configurar sua chave da API do X (Twitter) como uma variável de ambiente no seu sistema.

### Windows PowerShell:
1. Abra o PowerShell e execute:
   ```powershell
   setx X_BEARER_TOKEN "COLE_SEU_TOKEN_AQUI"
   ```
2. **REINICIE O VSCODE / JUPYTER** para que a nova variável seja carregada.

In [1]:
# CÉLULA 1 — INSTALAÇÃO DE DEPENDÊNCIAS
!pip install pandas numpy tqdm python-dotenv tweepy spacy nltk
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     --------------------------------------- 0.0/12.8 MB 217.9 kB/s eta 0:00:59
     --------------------------------------- 0.0/12.8 MB 326.8 kB/s eta 0:00:40
     --------------------------------------- 0.1/12.8 MB 465.5 kB/s eta 0:00:28
     --------------------------------------- 0.1/12.8 MB 654.9 kB/s eta 0:00:20
      --------------------------------------- 0.3/12.8 MB 1.1 MB/s eta 0:00:12
     - -------------------------------------- 0.3/12.8 MB 1.2 MB/s eta 0:00:11
     - -------------------------------------- 0.5/12.8 MB 1.9 MB/s eta 0:00:07
     -- ------------------------------------- 0.8/12.8 MB 2.4 MB/s eta 0:00:05
     --- ------------------------------------ 1.1/12.8 MB 2.9 MB/s eta 0:00:05
     ---- ----------------------------------- 1.4/12.8 MB 3.2 MB/s eta 0:00:04
     ------ --------------------------------- 2.1/12.8 MB 4.4 MB/s eta 0:00:03
     --------- ------------------------------ 3.0/12.8 

In [2]:
# CÉLULA 2 — IMPORTS E CONFIG
import os
import pandas as pd
import numpy as np
import logging
import time
import re
from datetime import datetime
import tweepy
import spacy
from tqdm.notebook import tqdm

# Configurar diretório de saída
OUTPUT_DIR = "outputs"
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# Configurar Logging
LOG_FILE = os.path.join(OUTPUT_DIR, "diario_bordo.log")
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(LOG_FILE, encoding='utf-8'),
        logging.StreamHandler()
    ]
)
logging.info("Notebook iniciado.")

2026-03-26 16:19:29,030 - INFO - Notebook iniciado.


In [3]:
# CÉLULA 3 — CARREGAR E LIMPAR companies.csv
def load_companies(path):
    logging.info("Lendo companies.csv...")
    try:
        df = pd.read_csv(path)
    except:
        df = pd.read_csv(path, encoding='latin1')
        
    # Limpeza básica
    df['funding_total_usd'] = pd.to_numeric(df['funding_total_usd'], errors='coerce').fillna(0)
    
    # Filtrar twitter válido
    df_clean = df[df['twitter_username'].notna() & (df['twitter_username'] != '')].copy()
    
    # Ordenar por funding
    df_clean = df_clean.sort_values(by='funding_total_usd', ascending=False)
    
    # Salvar filtrado
    df_clean.to_csv(os.path.join(OUTPUT_DIR, "companies_top_funding_with_twitter.csv"), index=False)
    
    print(f"Total original: {len(df)}")
    print(f"Total com Twitter: {len(df_clean)}")
    return df_clean

df_companies = load_companies("companies.csv")
df_companies.head(10)

2026-03-26 16:19:30,562 - INFO - Lendo companies.csv...


Total original: 196553
Total com Twitter: 80591


,id,Unnamed: 0.1,entity_type,entity_id,parent_id,name,normalized_name,permalink,category_code,status,...,first_milestone_at,last_milestone_at,milestones,relationships,created_by,created_at,updated_at,lat,lng,ROI
97784,c:242735,97784,Company,242735,NaN,sigmacare,sigmacare,/company/sigmacare,health,operating,...,NaN,NaN,NaN,2.0,prasant2013,2013-07-30 04:33:10,2013-07-30 04:33:10,40.712775,-74.005973,NaN
161551,c:5,161551,Company,5,NaN,Facebook,facebook,/company/facebook,social,ipo,...,2009-01-10,2013-12-12,5.0,269.0,initial-importer,2007-05-25 21:22:15,2013-08-27 14:41:51,37.452960,-122.181725,NaN
175825,c:64365,175825,Company,64365,NaN,Carestream,carestream,/company/carestream-health,biotech,operating,...,1991-01-01,1991-01-01,1.0,2.0,NaN,2010-12-19 10:34:00,2013-11-26 09:49:49,43.161030,-77.610922,NaN
81001,c:22568,81001,Company,22568,NaN,Solyndra,solyndra,/company/solyndra,manufacturing,operating,...,2010-01-01,2012-05-14,3.0,12.0,gene,2009-05-05 21:00:26,2013-03-24 10:19:15,37.548270,-121.988572,NaN
170979,c:5951,170979,Company,5951,NaN,Fisker Automotive,fisker automotive,/company/fisker,automotive,operating,...,2008-09-30,2012-08-01,3.0,9.0,joel,2008-05-28 04:32:49,2013-03-16 20:58:23,33.684567,-117.826505,NaN
151715,c:39799,151715,Company,39799,NaN,O3b Networks,o3b networks,/company/o3b-networks,enterprise,operating,...,2008-01-01,2008-01-01,1.0,21.0,bibilyana,2010-01-14 12:52:14,2013-04-05 20:01:42,18.336811,-64.728095,NaN
2971,c:12,2971,Company,12,NaN,Twitter,twitter,/company/twitter,social,ipo,...,2010-09-01,2013-12-12,6.0,162.0,initial-importer,2007-06-01 08:42:34,2013-08-27 14:41:20,37.774929,-122.419415,NaN
2225,c:11391,2225,Company,11391,NaN,Groupon,groupon,/company/groupon,web,ipo,...,2013-02-01,2013-11-01,3.0,101.0,NaN,2008-09-23 22:07:32,2013-08-22 19:37:33,41.878114,-87.629798,NaN
181834,c:7060,181834,Company,7060,NaN,Xerox,xerox,/company/xerox,hardware,ipo,...,2010-06-16,2013-01-11,5.0,112.0,NaN,2008-06-23 15:43:20,2012-08-04 09:42:04,33.902237,-118.081733,NaN
159845,c:4825,159845,Company,4825,NaN,"Sirius XM Radio, Inc.",sirius xm radio,/company/sirius,public_relations,ipo,...,2009-01-01,2013-07-09,4.0,11.0,mark,2008-04-29 20:37:12,2012-07-09 17:00:15,NaN,NaN,NaN


In [4]:
# CÉLULA 4 — FUNÇÕES PARA COLETA NA X API (Tweepy v2)
def get_client():
    token = os.environ.get("X_BEARER_TOKEN")
    if not token:
        raise ValueError("X_BEARER_TOKEN não definida! Veja a Célula 0.")
    return tweepy.Client(bearer_token=token, wait_on_rate_limit=True)

def collect_tweets(client, username, max_tweets=100):
    try:
        user = client.get_user(username=username)
        if not user.data: return []
        
        tweets_data = []
        # Coleta de 100 tweets (uma página de 100)
        response = client.get_users_tweets(
            id=user.data.id, 
            max_results=100, 
            tweet_fields=["created_at","public_metrics","lang"]
        )
        
        if response.data:
            for t in response.data:
                tweets_data.append({
                    "tweet_id": t.id,
                    "text": t.text,
                    "created_at": t.created_at,
                    "lang": t.lang,
                    "likes": t.public_metrics['like_count'],
                    "retweets": t.public_metrics['retweet_count']
                })
        return tweets_data
    except Exception as e:
        logging.error(f"Erro para {username}: {e}")
        return []

In [5]:
# CÉLULA 5 — EXECUÇÃO DA COLETA
N_STARTUPS = 50
TWEETS_PER_STARTUP = 100
client = get_client()

all_tweets = []
for idx, row in tqdm(df_companies.iloc[:N_STARTUPS].iterrows(), total=N_STARTUPS):
    uname = row['twitter_username']
    res = collect_tweets(client, uname, TWEETS_PER_STARTUP)
    for r in res:
        r['company_name'] = row['name']
        all_tweets.append(r)
    
    # Salvar incrementalmente a cada 5 startups
    if idx % 5 == 0:
        pd.DataFrame(all_tweets).to_csv(os.path.join(OUTPUT_DIR, "tweets_raw.csv"), index=False)

df_tweets = pd.DataFrame(all_tweets)
df_tweets.to_csv(os.path.join(OUTPUT_DIR, "tweets_raw.csv"), index=False)
print(f"Total de tweets coletados: {len(df_tweets)}")

ImportError: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html

In [ ]:
# CÉLULA 6 — ETAPA 1 (Regex + spaCy)
nlp = spacy.load("en_core_web_sm")

def analyze_narrative(df):
    patterns = {
        'problem': r'problem|gap|challenge|pain',
        'solution': r'solution|platform|tool|service',
        'mission': r'mission|vision|goal|purpose',
        'impact': r'impact|transform|help|improve'
    }
    
    # Concatenar descrições
    cols = ['description', 'short_description', 'overview']
    df['text_ana'] = df[df.columns.intersection(cols)].fillna('').agg(' '.join, axis=1)
    
    results = []
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        text = row['text_ana'].lower()
        features = {f"has_{k}": (1 if re.search(v, text) else 0) for k, v in patterns.items()}
        
        # spaCy NER
        doc = nlp(row['text_ana'][:10000])
        ents = [e.label_ for e in doc.ents]
        features['n_org'] = ents.count('ORG')
        features['n_person'] = ents.count('PERSON')
        features['n_gpe'] = ents.count('GPE')
        
        # Score
        score = sum([features[f"has_{k}"] for k in patterns.keys()]) + (features['n_org'] * 0.1)
        features['narrative_score'] = round(score, 2)
        features['name'] = row['name']
        features['funding_total_usd'] = row['funding_total_usd']
        results.append(features)
        
    return pd.DataFrame(results)

df_features = analyze_narrative(df_companies.head(100))
df_features.to_csv(os.path.join(OUTPUT_DIR, "etapa1_narrative_features.csv"), index=False)
print("Top 15 por Narrative Score:")
df_features.sort_values(by='narrative_score', ascending=False).head(15)

ImportError: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html

: 

In [ ]:
# CÉLULA 7 — EXPORTAR PARA .PY
!jupyter nbconvert --to script geral.ipynb --output outputs/projeto_final_exportado